## Create tensors for estimation
- Position
- Velocity
- Frame size
- FOV classification ground truth
- Meta values: [brand_id, color_id, yaw, vx, vy, x, y, rows]

# Meta values are encoded as follows for every experiment

## Vehicle Brand/Model
0. Mercedes Sprinter
1. Nissan Patrol
2. Tesla Model3

## Vehicle color
0. Black
1. Gray 
2. White

## Yaw
- 90
- -90
- 0
- 180

## Rows 
- Gives the effective length of the experiment
- For every experiment vectors are created with respect to the longest experiment

# First move the new dataset values into the folders with naming sample_i
first_sample_idx = 1010

# naming convention
vehicle.mercedes.sprinter_Black_vx-5.838_vy0.000_x-14.162_y17.000_yaw180.0_lane0_randomoffset0.00

# meta values
brand: mercedes \
color: black \
yaw: 180.0 \
vx: -10.525 \
vy: 0.000 \
x: -9.475 \
y: 17.000 \
rows: 270 \
source_folder: vehicle.mercedes.sprinter_Black_vx-10.525_vy0.000_x-9.475_y17.000_yaw180.0


## Configuration Settings


In [1]:
# JUPYTER CELL — Copy CSVs into sample_i folders, create meta.txt, and report row counts

from pathlib import Path
import re, csv, shutil, json
from typing import List

# --- CONFIG: set these before running ---
# NOTE FROM ISAAC: Gaofeng changed the way he generated datasets between seeeds 1 and 2. If you see an error, go to the cell and check for comments by me (to change column indexing, regex strings, etc)

seed_identifier = "seed1"
# seed_identifier = "seed2"
carla_run_identifier = "separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy"
# network_configuration = "bw10mbit_delay0_jitter0_loss0"
network_configuration = "bw100mbit_delay20_jitter5_loss0"



f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/raw_files/framesizes"

FRAME_SRC_ROOT = Path(f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/raw_files/framesizes")     # directory containing long-named vehicle.* folders
PACKET_SRC_ROOT = Path(f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/raw_files/packetsizes/{network_configuration}") 
TRAJECTORY_SRC_ROOT = Path(f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/raw_files/trajectories")
print(FRAME_SRC_ROOT)
print(PACKET_SRC_ROOT)
print(TRAJECTORY_SRC_ROOT)

SAMPLES_DST_ROOT    = Path(f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/indexed_samples")   # destination directory to create sample_i folders
TENSORS_DST_ROOT    = Path(f"/home/gaofeng/zanoria/grayassets_datasets/{seed_identifier}/{carla_run_identifier}/tensors")   # destination directory to create sample_i folders

START_INDEX = 0                           # starting sample_i index
OVERWRITE   = True                       # overwrite existing sample_i folders if they exist
DO_FRAMESIZE_AND_TRAJECTORY = True          # whether or not to write the framesize and trajectory samples (they stay the same for each network configuration)
# ---------------------------------------

SAMPLES_DST_ROOT.mkdir(parents=True, exist_ok=True)

/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/raw_files/framesizes
/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/raw_files/packetsizes/bw100mbit_delay20_jitter5_loss0
/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/raw_files/trajectories


# Copy Framesize CSV's

In [2]:
# THIS IS WHERE THE NUMBERED SAMPLE CSVS FOR THE FRAMESIZES WILL GO
FRAMESIZES_DST_ROOT = SAMPLES_DST_ROOT / "framesizes"
FRAMESIZES_DST_ROOT.mkdir(parents=True, exist_ok=True)

FOLDER_RE = re.compile(
    r"""^vehicle\.
        (?P<brand>[^.]+)\.
        (?P<model>[^_]+)_
        (?P<color>[^_]+)_
        vx(?P<vx>[-+]?\d+(?:\.\d+)?)_
        vy(?P<vy>[-+]?\d+(?:\.\d+)?)_
        x(?P<x>[-+]?\d+(?:\.\d+)?)_
        y(?P<y>[-+]?\d+(?:\.\d+)?)_
        yaw(?P<yaw>[-+]?\d+(?:\.\d+)?)

        #TODO: YOU MAY NEED TO UNCOMMENT THE FOLLOWING 2 LINES BASED ON WHEN GAOFENG GENERATED THE DATASET

        # _lane(?P<lane>-?\d+)
        # _randomoffset(?P<offset>[-+]?\d+(?:\.\d+)?)
        $""",
    re.VERBOSE
)

def _is_numeric_cell(s: str) -> bool:
    try:
        float(s)
        return True
    except Exception:
        return False

def _count_data_rows(csv_path: Path) -> int:
    """Counts data rows, skipping the first row if it looks like a header (any non-numeric cell)."""
    with csv_path.open("r", newline="") as f:
        reader = csv.reader(f)
        try:
            first = next(reader)
        except StopIteration:
            return 0
        header_like = any(not _is_numeric_cell(c) for c in first)
        count = 0 if header_like else 1
        for _ in reader:
            count += 1
        return count

def _gather_csvs(src_dir: Path) -> List[Path]:
    return sorted([p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() == ".csv"])

candidates = sorted([p for p in FRAME_SRC_ROOT.iterdir() if p.is_dir()])
i = START_INDEX
processed = 0

summary_rows = []   # collect per-sample info for reporting
global_max_csv_rows = 0
global_max_sample_rows = 0

for folder in candidates:
    m = FOLDER_RE.match(folder.name)
    if not m:
        continue  # skip non-matching folders

    info = m.groupdict()
    brand = info["brand"].lower()
    color = info["color"].lower()
    yaw = float(info["yaw"])
    vx = float(info["vx"])
    vy = float(info["vy"])
    x  = float(info["x"])
    y  = float(info["y"])
    source_folder_name = folder.name

    csv_files = _gather_csvs(folder)
    if len(csv_files) == 0:
        print(f"[WARN] No CSV files in {folder}. Skipping.")
        i += 1
        continue
    if len(csv_files) != 4:
        print(f"[WARN] Expected 4 CSVs in {folder}, found {len(csv_files)}. Proceeding with what’s there.")

    row_counts = [_count_data_rows(p) for p in csv_files]
    # choose 'rows' for meta.txt: use the minimum across CSVs (robust if lengths differ)
    rows_for_meta = min(row_counts) if row_counts else 0

    # track maxima
    if row_counts:
        local_max = max(row_counts)
        if local_max > global_max_csv_rows:
            global_max_csv_rows = local_max
    if rows_for_meta > global_max_sample_rows:
        global_max_sample_rows = rows_for_meta

    sample_dir = FRAMESIZES_DST_ROOT / f"sample_{i}"
    status = "created"
    if sample_dir.exists():
        if OVERWRITE:
            shutil.rmtree(sample_dir)
        else:
            print(f"[WARN] {sample_dir} exists and OVERWRITE=False. Skipping copy/meta (reporting counts only).")
            status = "skipped_exists"

    # create/copy/write meta if not skipped
    if status != "skipped_exists":
        sample_dir.mkdir(parents=True, exist_ok=True)
        for p in csv_files:
            shutil.copy2(p, sample_dir / p.name)

        meta_text = (
            f"brand: {brand}\n"
            f"color: {color}\n"
            f"yaw: {yaw}\n"
            f"vx: {vx}\n"
            f"vy: {vy}\n"
            f"x: {x}\n"
            f"y: {y}\n"
            f"rows: {rows_for_meta}\n"
            f"source_folder: {source_folder_name}\n"
        )
        (sample_dir / "meta.txt").write_text(meta_text)

    print(f"[OK] {folder.name} -> sample_{i}  "
          f"(rows_for_meta={rows_for_meta}, csv_row_counts={row_counts}, status={status})")

    summary_rows.append({
        "sample_i": i,
        "source_folder": source_folder_name,
        "sample_dir": str(sample_dir),
        "rows_for_meta": rows_for_meta,
        "csv_row_counts": row_counts,
        "all_equal": len(set(row_counts)) == 1,
        "status": status,
    })

    i += 1
    processed += 1

# --- reporting in the notebook ---
print(f"\nDone. Processed {processed} matching folder(s). Output root: {FRAMESIZES_DST_ROOT}")
print(f"Max rows across all CSV files: {global_max_csv_rows}")
print(f"Max 'rows' written to meta.txt across samples: {global_max_sample_rows}")

# Pretty display with pandas if available
try:
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    display(df)
except Exception:
    # Fallback: print JSON lines
    for rec in summary_rows:
        print(json.dumps(rec, indent=2))


[OK] vehicle.mercedes.sprinter_Black_vx-10.525_vy0.000_x-9.475_y17.000_yaw180.0 -> sample_0  (rows_for_meta=269, csv_row_counts=[269, 269, 269, 269], status=created)
[OK] vehicle.mercedes.sprinter_Black_vx-10.670_vy0.000_x-9.330_y17.000_yaw180.0 -> sample_1  (rows_for_meta=269, csv_row_counts=[269, 269, 269, 269], status=created)
[OK] vehicle.mercedes.sprinter_Black_vx-10.700_vy0.000_x-9.300_y17.000_yaw180.0 -> sample_2  (rows_for_meta=269, csv_row_counts=[269, 269, 269, 269], status=created)
[OK] vehicle.mercedes.sprinter_Black_vx-12.083_vy0.000_x-7.917_y17.000_yaw180.0 -> sample_3  (rows_for_meta=239, csv_row_counts=[239, 239, 239, 239], status=created)
[OK] vehicle.mercedes.sprinter_Black_vx-12.972_vy0.000_x-7.028_y17.000_yaw180.0 -> sample_4  (rows_for_meta=239, csv_row_counts=[239, 239, 239, 239], status=created)
[OK] vehicle.mercedes.sprinter_Black_vx-13.042_vy0.000_x-6.958_y17.000_yaw180.0 -> sample_5  (rows_for_meta=239, csv_row_counts=[239, 239, 239, 239], status=created)
[OK]

[OK] vehicle.mercedes.sprinter_White_vx15.170_vy0.000_x-105.170_y24.000_yaw0.0 -> sample_319  (rows_for_meta=209, csv_row_counts=[209, 209, 209, 209], status=created)
[OK] vehicle.mercedes.sprinter_White_vx16.302_vy0.000_x-106.302_y24.000_yaw0.0 -> sample_320  (rows_for_meta=209, csv_row_counts=[209, 209, 209, 209], status=created)
[OK] vehicle.mercedes.sprinter_White_vx17.530_vy0.000_x-107.530_y24.000_yaw0.0 -> sample_321  (rows_for_meta=179, csv_row_counts=[179, 179, 179, 179], status=created)
[OK] vehicle.mercedes.sprinter_White_vx17.657_vy0.000_x-107.657_y24.000_yaw0.0 -> sample_322  (rows_for_meta=179, csv_row_counts=[179, 179, 179, 179], status=created)
[OK] vehicle.mercedes.sprinter_White_vx19.103_vy0.000_x-109.103_y24.000_yaw0.0 -> sample_323  (rows_for_meta=179, csv_row_counts=[179, 179, 179, 179], status=created)
[OK] vehicle.mercedes.sprinter_White_vx19.731_vy0.000_x-109.731_y24.000_yaw0.0 -> sample_324  (rows_for_meta=179, csv_row_counts=[179, 179, 179, 179], status=created

,sample_i,source_folder,sample_dir,rows_for_meta,csv_row_counts,all_equal,status
0,0,vehicle.mercedes.sprinter_Black_vx-10.525_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,269,"[269, 269, 269, 269]",True,created
1,1,vehicle.mercedes.sprinter_Black_vx-10.670_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,269,"[269, 269, 269, 269]",True,created
2,2,vehicle.mercedes.sprinter_Black_vx-10.700_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,269,"[269, 269, 269, 269]",True,created
3,3,vehicle.mercedes.sprinter_Black_vx-12.083_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,239,"[239, 239, 239, 239]",True,created
4,4,vehicle.mercedes.sprinter_Black_vx-12.972_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,239,"[239, 239, 239, 239]",True,created
...,...,...,...,...,...,...,...
1005,1005,vehicle.tesla.model3_White_vx8.442_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,329,"[329, 329, 329, 329]",True,created
1006,1006,vehicle.tesla.model3_White_vx8.828_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,299,"[299, 299, 299, 299]",True,created
1007,1007,vehicle.tesla.model3_White_vx9.563_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,299,"[299, 299, 299, 299]",True,created
1008,1008,vehicle.tesla.model3_White_vx9.610_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,299,"[299, 299, 299, 299]",True,created


## Copy Raw Packetsize CSV's

In [3]:
PACKETSIZES_DST_ROOT = SAMPLES_DST_ROOT / "packetsizes" / network_configuration
PACKETSIZES_DST_ROOT.mkdir(parents=True, exist_ok=True)


def _is_numeric_cell(s: str) -> bool:
    try:
        float(s)
        return True
    except Exception:
        return False

def _count_data_rows(csv_path: Path) -> int:
    """Counts data rows, skipping the first row if it looks like a header (any non-numeric cell)."""
    with csv_path.open("r", newline="") as f:
        reader = csv.reader(f)
        try:
            first = next(reader)
        except StopIteration:
            return 0
        header_like = any(not _is_numeric_cell(c) for c in first)
        count = 0 if header_like else 1
        for _ in reader:
            count += 1
        return count

def _gather_csvs(src_dir: Path) -> List[Path]:
    return sorted([p for p in src_dir.iterdir() if p.is_file() and p.suffix.lower() == ".csv"])

candidates = sorted([p for p in PACKET_SRC_ROOT.iterdir() if p.is_dir()])
print(candidates)
i = START_INDEX
processed = 0

summary_rows = []   # collect per-sample info for reporting
global_max_csv_rows = 0
global_max_sample_rows = 0

for folder in candidates:
    m = FOLDER_RE.match(folder.name)
    if not m:
        continue  # skip non-matching folders

    info = m.groupdict()
    brand = info["brand"].lower()
    color = info["color"].lower()
    yaw = float(info["yaw"])
    vx = float(info["vx"])
    vy = float(info["vy"])
    x  = float(info["x"])
    y  = float(info["y"])
    source_folder_name = folder.name

    csv_files = _gather_csvs(folder)
    if len(csv_files) == 0:
        print(f"[WARN] No CSV files in {folder}. Skipping.")
        i += 1
        continue
    if len(csv_files) != 4:
        print(f"[WARN] Expected 4 CSVs in {folder}, found {len(csv_files)}. Proceeding with what’s there.")

    row_counts = [_count_data_rows(p) for p in csv_files]
    # choose 'rows' for meta.txt: use the minimum across CSVs (robust if lengths differ)

    # track maxima
    if row_counts:
        local_max = max(row_counts)
        if local_max > global_max_csv_rows:
            global_max_csv_rows = local_max

    sample_dir = PACKETSIZES_DST_ROOT / f"sample_{i}"
    status = "created"

    # create/copy/write meta if not skipped
    if status != "skipped_exists":
        sample_dir.mkdir(parents=True, exist_ok=True)
        for p in csv_files:
            shutil.copy2(p, sample_dir / p.name)


    print(f"[OK] {folder.name} -> sample_{i}  "
          f"(csv_row_counts={row_counts}, status={status})")

    summary_rows.append({
        "sample_i": i,
        "source_folder": source_folder_name,
        "sample_dir": str(sample_dir),
        "csv_row_counts": row_counts,
        "all_equal": len(set(row_counts)) == 1,
        "status": status,
    })

    i += 1
    processed += 1

# --- reporting in the notebook ---
print(f"\nDone. Processed {processed} matching folder(s). Output root: {PACKETSIZES_DST_ROOT}")
print(f"Max rows across all CSV files: {global_max_csv_rows}")
# print(f"Max 'rows' written to meta.txt across samples: {global_max_sample_rows}")

# Pretty display with pandas if available
try:
    import pandas as pd
    df = pd.DataFrame(summary_rows)
    display(df)
except Exception:
    # Fallback: print JSON lines
    for rec in summary_rows:
        print(json.dumps(rec, indent=2))


[PosixPath('/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/raw_files/packetsizes/bw100mbit_delay20_jitter5_loss0/vehicle.mercedes.sprinter_Black_vx-10.525_vy0.000_x-9.475_y17.000_yaw180.0'), PosixPath('/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/raw_files/packetsizes/bw100mbit_delay20_jitter5_loss0/vehicle.mercedes.sprinter_Black_vx-10.670_vy0.000_x-9.330_y17.000_yaw180.0'), PosixPath('/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/raw_files/packetsizes/bw100mbit_delay20_jitter5_loss0/vehicle.mercedes.sprinter_Black_vx-10.700_vy0.000_x-9.300_y17.000_yaw180.0'), PosixPath('/home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_re

,sample_i,source_folder,sample_dir,csv_row_counts,all_equal,status
0,0,vehicle.mercedes.sprinter_Black_vx-10.525_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[5253, 6743, 5973, 6887]",False,created
1,1,vehicle.mercedes.sprinter_Black_vx-10.670_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[5250, 6745, 5976, 6881]",False,created
2,2,vehicle.mercedes.sprinter_Black_vx-10.700_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[5185, 6697, 5936, 6863]",False,created
3,3,vehicle.mercedes.sprinter_Black_vx-12.083_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[4596, 5950, 5283, 6096]",False,created
4,4,vehicle.mercedes.sprinter_Black_vx-12.972_vy0....,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[4585, 5953, 5262, 6083]",False,created
...,...,...,...,...,...,...
1005,1005,vehicle.tesla.model3_White_vx8.442_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[6244, 8129, 7126, 8440]",False,created
1006,1006,vehicle.tesla.model3_White_vx8.828_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[5754, 7436, 6522, 7671]",False,created
1007,1007,vehicle.tesla.model3_White_vx9.563_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[5754, 7431, 6499, 7685]",False,created
1008,1008,vehicle.tesla.model3_White_vx9.610_vy0.000_x-9...,/home/gaofeng/zanoria/grayassets_datasets/seed...,"[5764, 7440, 6508, 7675]",False,created


## Create aggregated packetsize CSV's from raw pktsizes

In [4]:
import numpy as np
import pandas as pd

# --- CONFIG ---
# GLOB_PATTERN = "video_camera*_pktsize_1x.csv"
GLOB_PATTERN = "video_camera*_pkts_1x_pktlen.csv"
SIGMA = 1  # split when gap > mean(dt) + SIGMA*std(dt)
CONST_TIME_THRESHOLD = .002
USE_CONST = False
AGGREGATION_WINDOW_LENGTH = 0.03333333333

def _read_pkts_csv(csv_path: Path) -> pd.DataFrame:
    # Peek the first line to see if there's a header
    with csv_path.open("r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip().lower()
    has_header = ("frame.time_relative" in first_line) and ("frame.len" in first_line)

    df = pd.read_csv(
        csv_path,
        sep=",",
        header=0 if has_header else None,
        names=None if has_header else ["frame.time_relative", "frame.len"],
        
        # MAY NEED TO USE COLUMNS [0,1], [1,2] OR [2,3] BASED ON WHEN GAOFENG WROTE THE DATASET
        # usecols=[0, 1],
        usecols=[1, 2],
        # usecols=[2, 3],

        comment="#",
        on_bad_lines="skip",
        dtype=str,
        encoding="utf-8-sig",
        skip_blank_lines=True,
    )

    # Coerce to numeric & clean
    # print(df.columns)
    for c in ["frame.time_relative", "frame.len"]:
        df[c] = pd.to_numeric(df[c].astype(str).str.strip(), errors="coerce")

    bad_mask = df[["frame.time_relative", "frame.len"]].isna().any(axis=1)
    dropped = int(bad_mask.sum())
    df = df[~bad_mask].sort_values("frame.time_relative").reset_index(drop=True)

    if dropped and not has_header:
        print(f"[info] Dropped {dropped} malformed row(s) in {csv_path.name}")

    return df

def _aggregate_by_time_gaps(df: pd.DataFrame, k_sigma: float = SIGMA, const_threshold: float = CONST_TIME_THRESHOLD, use_const_threshold: bool=USE_CONST) -> pd.DataFrame:
    """
    Split into bursts when Δt > mean(Δt) + k_sigma * std(Δt).
    Output:
      packet_burst.time_relative (first packet time in burst)
      pkt_frame_sum.len          (sum of frame.len over burst)
    """
    t = df["frame.time_relative"].to_numpy()
    fl = df["frame.len"].to_numpy()

    if t.size == 0:
        return pd.DataFrame(columns=["packet_burst.time_relative", "pkt_frame_sum.len"])
    if t.size == 1:
        return pd.DataFrame({
            "packet_burst.time_relative": [float(t[0])],
            "pkt_frame_sum.len": [float(fl[0])],
        })

    dt = np.diff(t)
    mean_dt = float(np.mean(dt))
    std_dt = float(np.std(dt, ddof=0))
    if use_const_threshold:
        thr = const_threshold
    else:
        thr = mean_dt + k_sigma * std_dt

    bursts = []
    start = 0
    for i in range(1, t.size):
        if (t[i] - t[i-1]) > thr:
            bursts.append((start, i-1))
            start = i
    bursts.append((start, t.size - 1))

    out = []
    for s, e in bursts:
        out.append({
            "packet_burst.time_relative": float(t[s]),
            "pkt_frame_sum.len": float(np.sum(fl[s:e+1])),
        })
    agg = pd.DataFrame(out)
    # Keep some stats as attrs (pre-pad)
    agg.attrs["threshold_seconds"] = thr
    agg.attrs["mean_dt_seconds"] = mean_dt
    agg.attrs["std_dt_seconds"] = std_dt
    return agg

def _aggregate_by_time_window(df: pd.DataFrame, window_len: float, anchor: str = "first") -> pd.DataFrame:
    """
    Aggregate packets into fixed, non-overlapping windows of length `window_len` (seconds),
    summing frame.len within each window.

    Parameters
    ----------
    df : DataFrame with columns ["frame.time_relative", "frame.len"], sorted by time.
    window_len : float > 0, window length in seconds (e.g., 0.0333333 for ~30 Hz).
    anchor : {"first", "zero", "min"}
        - "first": windows start at the first packet's timestamp (default).
        - "zero" : windows start at t=0.0
        - "min"  : windows start at the minimum timestamp in df.

    Returns
    -------
    DataFrame with columns:
      - "packet_burst.time_relative": start time of each window
      - "pkt_frame_sum.len"        : sum of frame.len within that window

    Notes
    -----
    - Produces *all* consecutive windows from anchor t0 up to the last packet's window,
      including windows that may contain zero packets (sum == 0).
    - Stores metadata in .attrs: window_seconds, anchor_t0, n_windows.
    """
    if window_len is None or window_len <= 0:
        raise ValueError("window_len must be a positive float (seconds)")

    t = df["frame.time_relative"].to_numpy(dtype=float)
    fl = df["frame.len"].to_numpy(dtype=float)

    if t.size == 0:
        out = pd.DataFrame(columns=["packet_burst.time_relative", "pkt_frame_sum.len"])
        out.attrs["window_seconds"] = float(window_len)
        out.attrs["anchor_t0"] = float("nan")
        out.attrs["n_windows"] = 0
        return out

    if anchor == "zero":
        t0 = 0.0
    elif anchor == "min":
        t0 = float(np.min(t))
    else:  # "first" (default)
        t0 = float(t[0])

    # Compute window indices for each packet (0,1,2,...) relative to t0
    # clamp negatives defensively if t0 > min(t)
    idx = np.floor((t - t0) / float(window_len)).astype(np.int64)
    idx = np.maximum(idx, 0)

    # Sum frame lengths per window index using bincount
    nwin = int(idx.max()) + 1
    sums = np.bincount(idx, weights=fl, minlength=nwin).astype(float)

    # Build output timeline at window starts
    starts = t0 + window_len * np.arange(nwin, dtype=float)

    out = pd.DataFrame({
        "packet_burst.time_relative": starts,
        "pkt_frame_sum.len": sums
    })

    # Metadata
    out.attrs["window_seconds"] = float(window_len)
    out.attrs["anchor_t0"] = float(t0)
    out.attrs["n_windows"] = int(nwin)

    return out

def _dest_name(csv_path: Path) -> Path:
    """..._pkts_1x_pktlen.csv -> ..._pktsize_windowed.csv"""
    s = csv_path.name
    m = re.search(r"_pkts_1x_pktlen\.csv$", s, flags=re.IGNORECASE)
    if m:
        new_name = re.sub(r"_pkts_1x_pktlen\.csv$", "_pktsize_windowed.csv", s, flags=re.IGNORECASE)
    else:
        stem = csv_path.stem
        new_name = f"{stem}_aggregated.csv"
        if csv_path.suffix:
            new_name = f"{stem}_aggregated{csv_path.suffix}"
    return csv_path.with_name(new_name)

def _avg_dt_from_agg(agg: pd.DataFrame) -> float:
    """Average Δt between burst start times in this file; NaN if <2 bursts."""
    if len(agg) < 2:
        return float("nan")
    dt = np.diff(agg["packet_burst.time_relative"].to_numpy(dtype=float))
    return float(np.mean(dt))

def _pad_to_length(agg: pd.DataFrame, target_len: int, avg_dt: float) -> pd.DataFrame:
    """End-pad a shorter aggregated series to target_len using last value + avg_dt spacing."""
    n = len(agg)
    if n >= target_len:
        return agg

    out = agg.copy()
    last_t = float(out["packet_burst.time_relative"].iloc[-1]) if n else 0.0
    last_v = float(out["pkt_frame_sum.len"].iloc[-1]) if n else 0.0

    # Guard: ensure a positive, finite avg_dt
    if not np.isfinite(avg_dt) or avg_dt <= 0.0:
        # Fallback to a tiny positive step to keep timestamps strictly increasing
        avg_dt = 1e-6

    k = target_len - n
    add_times = last_t + avg_dt * np.arange(1, k + 1, dtype=float)
    add_vals = np.full(k, last_v, dtype=float)

    pad_df = pd.DataFrame({
        "packet_burst.time_relative": add_times,
        "pkt_frame_sum.len": add_vals,
    })
    padded = pd.concat([out, pad_df], ignore_index=True)
    padded.attrs.update(agg.attrs)
    padded.attrs["padded"] = True
    padded.attrs["pad_count"] = k
    padded.attrs["pad_avg_dt_seconds"] = avg_dt
    return padded

def aggregate_all_samples(PACKETSIZES_DST_ROOT: Path = PACKETSIZES_DST_ROOT, glob_pattern: str = GLOB_PATTERN):
    sample_dirs = sorted([p for p in PACKETSIZES_DST_ROOT.glob("sample_*") if p.is_dir()])
    # print(sample_dirs)
    total_in, total_out = 0, 0
    warn_count = 0

    for sdir in sample_dirs:
        csvs = sorted(sdir.glob(glob_pattern))
        if not csvs:
            print("passing")
            continue

        # 1) Aggregate all files first (don’t write yet)
        aggs = {}
        burst_counts = {}
        per_file_avg_dt = {}

        for csv_path in csvs:
            total_in += 1
            df = _read_pkts_csv(csv_path)
            # agg = _aggregate_by_time_gaps(df, k_sigma=SIGMA)
            agg = _aggregate_by_time_window(df, window_len=AGGREGATION_WINDOW_LENGTH)
            aggs[csv_path] = agg
            burst_counts[csv_path.name] = len(agg)
            per_file_avg_dt[csv_path.name] = _avg_dt_from_agg(agg)

        # 2) Compute target length (max) and a fallback dt (median across files)
        target_len = max(burst_counts.values())
        valid_dts = [v for v in per_file_avg_dt.values() if np.isfinite(v) and v > 0.0]
        group_fallback_dt = float(np.median(valid_dts)) if valid_dts else 1e-6

        # 3) Report inconsistency (pre-pad) and then pad shorter ones
        unique_counts = set(burst_counts.values())
        if len(unique_counts) != 1:
            print(f"[WARN] Inconsistent burst counts in {sdir.relative_to(PACKETSIZES_DST_ROOT)} (pre-pad):")
            for fname, cnt in burst_counts.items():
                print(f"       • {fname}: {cnt} bursts")
            warn_count += 1
        else:
            only = next(iter(unique_counts))
            print(f"[OK] {sdir.relative_to(PACKETSIZES_DST_ROOT)} — all files have {only} bursts (pre-pad)")

        # 4) Pad and write each file
        for csv_path, agg in aggs.items():
            bcount_before = len(agg)
            avg_dt = per_file_avg_dt[csv_path.name]
            if not (np.isfinite(avg_dt) and avg_dt > 0.0):
                avg_dt = group_fallback_dt  # robust fallback if a file has <2 bursts

            padded = _pad_to_length(agg, target_len, avg_dt)
            out_path = _dest_name(csv_path)
            padded.to_csv(
                out_path,
                index=False,
                header=["packet_burst.time_relative", "pkt_frame_sum.len"],
                float_format="%.9f",
            )

            thr = agg.attrs.get("threshold_seconds", float("nan"))
            mu  = agg.attrs.get("mean_dt_seconds", float("nan"))
            sd  = agg.attrs.get("std_dt_seconds", float("nan"))
            if len(padded) > bcount_before:
                k = len(padded) - bcount_before
                print(
                    f"[OK] {csv_path.relative_to(PACKETSIZES_DST_ROOT)} -> {out_path.name}  "
                    f"bursts={bcount_before}→{len(padded)} (padded {k} with avg_dt={avg_dt:.9f}s)  "
                    f"thr={thr*1e6:.2f}µs (mean={mu*1e6:.2f}, std={sd*1e6:.2f})"
                )
            else:
                print(
                    f"[OK] {csv_path.relative_to(PACKETSIZES_DST_ROOT)} -> {out_path.name}  "
                    f"bursts={len(padded)}  thr={thr*1e6:.2f}µs (mean={mu*1e6:.2f}, std={sd*1e6:.2f})"
                )
            total_out += 1

        # 5) Post-pad verification
        post_counts = {}
        for p in csvs:
            out_path = _dest_name(p)
            try:
                tmp = pd.read_csv(out_path, usecols=[0,1])
            except Exception:
                tmp = aggs[p]  # fallback to memory copy
            post_counts[out_path.name] = len(tmp)
        post_unique = set(post_counts.values())
        if len(post_unique) == 1:
            only = next(iter(post_unique))
            print(f"[OK] {sdir.relative_to(PACKETSIZES_DST_ROOT)} — all files padded to {only} bursts")
        else:
            print(f"[WARN] {sdir.relative_to(PACKETSIZES_DST_ROOT)} — still inconsistent after padding: {post_counts}")

    print(f"\nDone. Aggregated {total_out}/{total_in} file(s). Output root: {PACKETSIZES_DST_ROOT}")
    print(f"Num of experiments with non-equal bursts found (pre-pad): {warn_count}")

# ---- run it ----
aggregate_all_samples(PACKETSIZES_DST_ROOT, GLOB_PATTERN)


[OK] sample_0 — all files have 269 bursts (pre-pad)
[OK] sample_0/video_camera1_pkts_1x_pktlen.csv -> video_camera1_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_0/video_camera2_pkts_1x_pktlen.csv -> video_camera2_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_0/video_camera3_pkts_1x_pktlen.csv -> video_camera3_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_0/video_camera4_pkts_1x_pktlen.csv -> video_camera4_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)


[OK] sample_0 — all files padded to 269 bursts
[OK] sample_1 — all files have 269 bursts (pre-pad)
[OK] sample_1/video_camera1_pkts_1x_pktlen.csv -> video_camera1_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_1/video_camera2_pkts_1x_pktlen.csv -> video_camera2_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_1/video_camera3_pkts_1x_pktlen.csv -> video_camera3_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_1/video_camera4_pkts_1x_pktlen.csv -> video_camera4_pktsize_windowed.csv  bursts=269  thr=nanµs (mean=nan, std=nan)
[OK] sample_1 — all files padded to 269 bursts
[OK] sample_10 — all files have 239 bursts (pre-pad)
[OK] sample_10/video_camera1_pkts_1x_pktlen.csv -> video_camera1_pktsize_windowed.csv  bursts=239  thr=nanµs (mean=nan, std=nan)
[OK] sample_10/video_camera2_pkts_1x_pktlen.csv -> video_camera2_pktsize_windowed.csv  bursts=239  thr=nanµs (mean=nan, std=nan)
[OK] sample_10/video_camera3_pk

### Plotting Test

In [5]:
test_plot = False
if test_plot:

    # csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed2/sample_0/video_camera1_pktsize_1x.csv"
    # csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed2/sample_856/video_camera3_pktsize_1x.csv"
    # csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed2/sample_105/video_camera1_pktsize_1x.csv"

    # SEED1 EXPERIMENTS
    csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed1/bw10_delay20_jitter5_loss0/sample_997/video_camera1_pktsize_1x.csv"
    # csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed1/bw30_delay0_jitter0_loss0/sample_997/video_camera1_pktsize_1x.csv"
    # csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed1/unencrypted_delay20_jitter5_loss0/sample_997/video_camera1_pktsize_1x.csv"

    import pandas as pd
    import matplotlib.pyplot as plt
    import numpy as np
    import matplotlib.ticker as ticker

    # Read CSV
    df = pd.read_csv(csv_path)

    # Make sure the columns are numeric and sorted by time
    df["frame.time_relative"] = pd.to_numeric(df["frame.time_relative"], errors="coerce")
    df["frame.len"] = pd.to_numeric(df["frame.len"], errors="coerce")
    df = df.dropna(subset=["frame.time_relative", "frame.len"]).sort_values("frame.time_relative")

    # Plot
    plt.figure(figsize=(9, 4.5))
    plt.plot(df["frame.time_relative"], df["frame.len"], marker=".", linestyle="-", linewidth=0.8, markersize=3)
    plt.xlabel("frame.time_relative (s)")
    plt.ylabel("frame.len (bytes)")
    plt.title("Packet payload size over capture time")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Load and prepare times
    # df = pd.read_csv(csv_path)
    t = pd.to_numeric(df["frame.time_relative"], errors="coerce").dropna().sort_values().to_numpy()
    dt_us = np.diff(t) * 1e6  # microseconds

    fig, ax = plt.subplots(figsize=(8, 4.5))
    n, bins, patches = ax.hist(dt_us, bins=100, edgecolor="black", linewidth=0.3)

    # ---- Make y-ticks more granular ----
    target_ticks = 10  # increase for denser ticks
    step = max(1, int(np.ceil(n.max() / target_ticks)//100*100))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(step))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))

    # Labels & style
    ax.set_xlabel("Inter-arrival Δt (µs)")
    ax.set_ylabel("Count")
    ax.set_title("Histogram of packet inter-arrival times")
    ax.grid(True, alpha=0.3, which="both", axis="y")
    fig.tight_layout()
    plt.show()

    # (Optional) quick stats
    print(f"N intervals: {dt_us.size}")
    print(f"Mean: {dt_us.mean():.2f} µs | Median: {np.median(dt_us):.2f} µs | Std: {dt_us.std():.2f} µs")

### Debug plots

In [6]:
if test_plot:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt

    # --- set your aggregated CSV path ---
    # csv_path = "./indexed_samples/seed2/sample_885/video_camera4_pktsize_aggregated.csv"
    # csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed1/bw30_delay0_jitter0_loss0/sample_997/video_camera1_pktsize_aggregated.csv"
    csv_path = r"/home/zanoria/IoBT/gray_assets/zanoria/carla/grouping_packets/indexed_samples/seed1/bw100_delay20_jitter5_loss0/sample_997/video_camera1_pktsize_aggregated.csv"

    # Load
    df = pd.read_csv(csv_path)
    if "pkt_frame_sum.len" not in df.columns:
        raise ValueError(f"'pkt_frame_sum.len' not found. Columns are: {list(df.columns)}")

    sizes = pd.to_numeric(df["pkt_frame_sum.len"], errors="coerce").dropna().to_numpy()

    # Choose a reasonable number of bins (Freedman–Diaconis rule)
    def fd_bins(x: np.ndarray) -> int:
        x = x[~np.isnan(x)]
        if x.size < 2:
            return 10
        iqr = np.subtract(*np.percentile(x, [75, 25]))
        if iqr <= 0:
            return max(10, int(np.sqrt(x.size)))
        width = 2 * iqr * (x.size ** (-1/3))
        if width <= 0:
            return max(10, int(np.sqrt(x.size)))
        return max(10, int(np.ceil((x.max() - x.min()) / width)))

    bins = fd_bins(sizes)

    # Plot
    plt.figure(figsize=(8, 4.5))
    plt.hist(sizes, bins=bins, edgecolor="black", linewidth=0.3)
    plt.xlabel("pkt_frame_sum.len (bytes)")
    plt.ylabel("Count")
    plt.title("Histogram of aggregated frame sizes")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4.5))
    plt.plot(df["packet_burst.time_relative"],df["pkt_frame_sum.len"])
    plt.xlabel("time")
    plt.ylabel("aggregated packets size")
    plt.title("Aggregated packets size time series")
    

    # (optional) quick stats
    print(f"N={sizes.size} | mean={sizes.mean():.1f} | median={np.median(sizes):.1f} | p95={np.percentile(sizes,95):.1f}")

    

## Create (and rename) trajectory CSV's

In [7]:
TRAJECTORIES_DST_ROOT = SAMPLES_DST_ROOT / "trajectories"
TRAJECTORIES_DST_ROOT.mkdir(parents=True, exist_ok=True)

# Check the ground truth position data and move them to sample_i as well
# For sample_i folder, start from the given index in the indexed samples folder
# Check the original naming convention go to the given directory and move the trajectories to sample_i

# JUPYTER CELL — Copy vehicle_trajectory.csv into sample_i folders based on meta.txt

from pathlib import Path
import re, shutil, json

# --- CONFIG: set these before running ---                        # overwrite vehicle_trajectory.csv if it already exists
SRC_FILENAME = "vehicle_trajectory.csv"           # source CSV filename expected in original folders
# ---------------------------------------

sample_pat = re.compile(r"^sample_(\d+)$")
source_line_pat = re.compile(r"^source_folder\s*:\s*(.+)$", re.IGNORECASE)

def _read_source_folder(meta_path: Path) -> str | None:
    if not meta_path.exists():
        return None
    for line in meta_path.read_text().splitlines():
        m = source_line_pat.match(line.strip())
        if m:
            return m.group(1).strip()
    return None

def _count_rows(csv_path: Path) -> int:
    # Simple row counter (counts all lines except an empty trailing line)
    try:
        with csv_path.open("r", encoding="utf-8", newline="") as f:
            return sum(1 for _ in f if _.strip() != "")
    except Exception:
        return -1  # indicate unreadable

# Discover sample_i folders and sort by numeric i
sample_dirs = []
for p in FRAMESIZES_DST_ROOT.iterdir():
    if p.is_dir():
        m = sample_pat.match(p.name)
        if m:
            i = int(m.group(1))
            if i >= START_INDEX:
                sample_dirs.append((i, p))
sample_dirs.sort(key=lambda t: t[0])

processed = 0
copied = 0
skipped = 0
records = []

for i, sample_dir in sample_dirs:
    # print(sample_dir)
    
    sample_name = sample_dir.name
    trajectory_out_dir = TRAJECTORIES_DST_ROOT / sample_name
    trajectory_out_dir.mkdir(parents=True, exist_ok=True)
    # print(sample_num)
    # continue

    meta_path = sample_dir / "meta.txt"
    source_folder_name = _read_source_folder(meta_path)
    if not source_folder_name:
        print(f"[WARN] sample_{i}: meta.txt missing or no 'source_folder' line. Skipping.")
        skipped += 1
        records.append({
            "sample_i": i, "sample_dir": str(sample_dir),
            "status": "missing_meta_or_source_field"
        })
        continue

    orig_dir = TRAJECTORY_SRC_ROOT / source_folder_name
    if not orig_dir.exists():
        print(f"[WARN] sample_{i}: original folder not found: {orig_dir}. Skipping.")
        skipped += 1
        records.append({
            "sample_i": i, "sample_dir": str(sample_dir),
            "source_folder": source_folder_name,
            "status": "original_folder_not_found"
        })
        continue

    src_csv = orig_dir / SRC_FILENAME
    if not src_csv.exists():
        print(f"[WARN] sample_{i}: {SRC_FILENAME} not found in {orig_dir}. Skipping.")
        skipped += 1
        records.append({
            "sample_i": i, "sample_dir": str(sample_dir),
            "source_folder": source_folder_name,
            "status": f"{SRC_FILENAME}_not_found"
        })
        continue

    dst_csv = trajectory_out_dir / SRC_FILENAME
    if dst_csv.exists() and not OVERWRITE:
        print(f"[INFO] sample_{i}: {SRC_FILENAME} already exists (overwrite=False). Skipping copy.")
        dst_rows = _count_rows(dst_csv)
        skipped += 1
        records.append({
            "sample_i": i, "sample_dir": str(sample_dir),
            "source_folder": source_folder_name,
            "status": "exists_skipped",
            "dst_rows": dst_rows
        })
        processed += 1
        continue

    # Ensure destination exists and copy
    sample_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_csv, dst_csv)
    dst_rows = _count_rows(dst_csv)
    print(f"[OK] sample_{i}: copied {SRC_FILENAME} (rows={dst_rows})")

    copied += 1
    processed += 1
    records.append({
        "sample_i": i, "sample_dir": str(sample_dir),
        "source_folder": source_folder_name,
        "status": "copied",
        "dst_rows": dst_rows
    })

print(f"\nDone. Iterated {len(sample_dirs)} sample folders (i >= {START_INDEX}). "
      f"Processed: {processed}, Copied: {copied}, Skipped: {skipped}")

# Optional: tabular summary if pandas is available
try:
    import pandas as pd
    df = pd.DataFrame(records).sort_values("sample_i")
    display(df)
    print(f"Max rows among experiments: {df['dst_rows'].max()}")
except Exception:
    for rec in records:
        print(json.dumps(rec, indent=2))



[OK] sample_0: copied vehicle_trajectory.csv (rows=271)
[OK] sample_1: copied vehicle_trajectory.csv (rows=271)
[OK] sample_2: copied vehicle_trajectory.csv (rows=271)
[OK] sample_3: copied vehicle_trajectory.csv (rows=241)
[OK] sample_4: copied vehicle_trajectory.csv (rows=241)
[OK] sample_5: copied vehicle_trajectory.csv (rows=241)
[OK] sample_6: copied vehicle_trajectory.csv (rows=241)
[OK] sample_7: copied vehicle_trajectory.csv (rows=241)
[OK] sample_8: copied vehicle_trajectory.csv (rows=241)
[OK] sample_9: copied vehicle_trajectory.csv (rows=241)
[OK] sample_10: copied vehicle_trajectory.csv (rows=241)
[OK] sample_11: copied vehicle_trajectory.csv (rows=241)
[OK] sample_12: copied vehicle_trajectory.csv (rows=211)
[OK] sample_13: copied vehicle_trajectory.csv (rows=211)
[OK] sample_14: copied vehicle_trajectory.csv (rows=211)
[OK] sample_15: copied vehicle_trajectory.csv (rows=211)
[OK] sample_16: copied vehicle_trajectory.csv (rows=211)
[OK] sample_17: copied vehicle_trajectory

[OK] sample_693: copied vehicle_trajectory.csv (rows=301)
[OK] sample_694: copied vehicle_trajectory.csv (rows=301)
[OK] sample_695: copied vehicle_trajectory.csv (rows=271)
[OK] sample_696: copied vehicle_trajectory.csv (rows=241)
[OK] sample_697: copied vehicle_trajectory.csv (rows=241)
[OK] sample_698: copied vehicle_trajectory.csv (rows=241)
[OK] sample_699: copied vehicle_trajectory.csv (rows=241)
[OK] sample_700: copied vehicle_trajectory.csv (rows=211)
[OK] sample_701: copied vehicle_trajectory.csv (rows=211)
[OK] sample_702: copied vehicle_trajectory.csv (rows=211)
[OK] sample_703: copied vehicle_trajectory.csv (rows=211)
[OK] sample_704: copied vehicle_trajectory.csv (rows=211)
[OK] sample_705: copied vehicle_trajectory.csv (rows=211)
[OK] sample_706: copied vehicle_trajectory.csv (rows=181)
[OK] sample_707: copied vehicle_trajectory.csv (rows=181)
[OK] sample_708: copied vehicle_trajectory.csv (rows=181)
[OK] sample_709: copied vehicle_trajectory.csv (rows=181)
[OK] sample_71

,sample_i,sample_dir,source_folder,status,dst_rows
0,0,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.mercedes.sprinter_Black_vx-10.525_vy0....,copied,271
1,1,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.mercedes.sprinter_Black_vx-10.670_vy0....,copied,271
2,2,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.mercedes.sprinter_Black_vx-10.700_vy0....,copied,271
3,3,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.mercedes.sprinter_Black_vx-12.083_vy0....,copied,241
4,4,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.mercedes.sprinter_Black_vx-12.972_vy0....,copied,241
...,...,...,...,...,...
1005,1005,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.tesla.model3_White_vx8.442_vy0.000_x-9...,copied,331
1006,1006,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.tesla.model3_White_vx8.828_vy0.000_x-9...,copied,301
1007,1007,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.tesla.model3_White_vx9.563_vy0.000_x-9...,copied,301
1008,1008,/home/gaofeng/zanoria/grayassets_datasets/seed...,vehicle.tesla.model3_White_vx9.610_vy0.000_x-9...,copied,301


Max rows among experiments: 481


# Create Tensors

## Create Position Tensors from CSV's

In [8]:
# Now for all samples from i = 0 to the end create tensors 
# Position, velocity, positions

# JUPYTER CELL — Build [E, T_FLAT, 3] position/velocity tensors from vehicle_trajectory.csv

from pathlib import Path
import re
import pandas as pd
import torch

# --- CONFIG (edit these) ---
CSV_NAME     = "vehicle_trajectory.csv"        # name inside each sample_i
T_FLAT       = 480                              # <- set desired fixed time length
DTYPE        = torch.float32                   # tensor dtype
# ---------------------------

# 1) Discover sample_i folders (i >= 0), map i -> Path
pat = re.compile(r"^sample_(\d+)$")
sample_dirs = {}
for p in TRAJECTORIES_DST_ROOT.iterdir():
    if p.is_dir():
        m = pat.match(p.name)
        if m:
            i = int(m.group(1))
            if i >= 0:
                sample_dirs[i] = p

if not sample_dirs:
    raise RuntimeError(f"No sample_i folders found in {TRAJECTORIES_DST_ROOT}")

max_idx = max(sample_dirs.keys())
E = max_idx + 1  # align index with sample_i (missing indices remain zero)

# 2) Allocate tensors
position = torch.zeros((E, T_FLAT, 3), dtype=DTYPE)
velocity = torch.zeros((E, T_FLAT, 3), dtype=DTYPE)

# Stats
missing_csv, bad_csv = [], []
n_truncated, n_short, n_filled = 0, 0, 0

# 3) Load and fill per sample
for i in range(E):
    sdir = sample_dirs.get(i, None)
    if sdir is None:
        # No such sample_i folder; remains zeros
        continue

    csv_path = sdir / CSV_NAME
    if not csv_path.exists():
        missing_csv.append(i)
        continue

    try:
        # Read only the needed columns
        df = pd.read_csv(csv_path, usecols=["x","y","z","vx","vy","vz"])
    except Exception as e:
        bad_csv.append((i, str(e)))
        continue

    # Effective timesteps we’ll fill for this sample
    t_eff = min(len(df), T_FLAT)
    if t_eff <= 0:
        continue

    # Prepare numpy slices (float32) and assign
    pos_np = df[["x","y","z"]].to_numpy(dtype="float32")[:t_eff]
    vel_np = df[["vx","vy","vz"]].to_numpy(dtype="float32")[:t_eff]

    position[i, :t_eff, :] = torch.from_numpy(pos_np)
    velocity[i, :t_eff, :] = torch.from_numpy(vel_np)
    n_filled += 1

    if len(df) > T_FLAT:
        n_truncated += 1
    elif len(df) < T_FLAT:
        n_short += 1

# 4) Report
print(f"Built tensors: E={E}, T_FLAT={T_FLAT}")
print(f"position.shape = {tuple(position.shape)}, velocity.shape = {tuple(velocity.shape)}")
print(f"Filled {n_filled} sample(s). Truncated: {n_truncated}, Shorter-than-T_FLAT: {n_short}")
print(f"Test print of the first position row: {position[0,:10,:]}")
if missing_csv:
    print(f"[WARN] Missing {CSV_NAME} for samples: {sorted(missing_csv)}")
if bad_csv:
    print("[WARN] Unreadable CSVs:")
    for i, err in bad_csv:
        print(f"  sample_{i}: {err}")


Built tensors: E=1010, T_FLAT=480
position.shape = (1010, 480, 3), velocity.shape = (1010, 480, 3)
Filled 1010 sample(s). Truncated: 0, Shorter-than-T_FLAT: 987
Test print of the first position row: tensor([[ -9.4746,  17.0000,   0.2932],
        [ -9.8254,  17.0000,   0.2864],
        [-10.1762,  17.0000,   0.2796],
        [-10.5270,  17.0000,   0.2728],
        [-10.8779,  17.0000,   0.2660],
        [-11.2287,  17.0000,   0.2592],
        [-11.5795,  17.0000,   0.2524],
        [-11.9303,  17.0000,   0.2456],
        [-12.2812,  17.0000,   0.2387],
        [-12.6320,  17.0000,   0.2319]])


## Create Framesize Tensor

In [9]:
# Similarly read the frame sizes for each sample
# Create a tensor of shape [E,T_max,4]


from pathlib import Path
import re
import pandas as pd
import torch

# --- CONFIG (edit these) ---
T_MAX        = 480                              # desired fixed timeline length
DTYPE        = torch.float32                   # tensor dtype
FILENAME_FMT = "video_camera{cam}_frame_typesize.csv"
# ---------------------------

# 1) Discover sample_i folders (i >= 0) and align E with the largest index
pat = re.compile(r"^sample_(\d+)$")
sample_dirs = {}
for p in FRAMESIZES_DST_ROOT.iterdir():
    if p.is_dir():
        m = pat.match(p.name)
        if m:
            i = int(m.group(1))
            if i >= 0:
                sample_dirs[i] = p

if not sample_dirs:
    raise RuntimeError(f"No sample_i folders found in {FRAMESIZES_DST_ROOT}")

E = max(sample_dirs.keys()) + 1  # tensor index aligns with sample_i (holes remain zeros)

# 2) Allocate the tensor [E, T_MAX, 4]
frame_sizes = torch.zeros((E, T_MAX, 4), dtype=DTYPE)

# Stats
missing_samples = []
missing_files = {1: [], 2: [], 3: [], 4: []}
bad_files = []
filled_count = 0
truncated = 0
shorter = 0

# 3) Fill per sample and per camera
for i in range(E):
    sdir = sample_dirs.get(i)
    if sdir is None:
        missing_samples.append(i)
        continue

    sample_filled = False
    for cam in (1, 2, 3, 4):
        csv_path = sdir / FILENAME_FMT.format(cam=cam)
        if not csv_path.exists():
            missing_files[cam].append(i)
            continue

        try:
            # Two columns, first is size, second is I/P (ignored). Handle whitespace or commas.
            df = pd.read_csv(
                csv_path,
                header=None,
                names=["size", "type"],
                sep=r"[\s,]+",
                engine="python",
                usecols=[0, 1],  # ensure we read only two columns
            )
            # Keep just numeric sizes; drop non-numerics safely
            sizes = pd.to_numeric(df["size"], errors="coerce").dropna().astype("float32").to_numpy()
        except Exception as e:
            bad_files.append((i, cam, str(e)))
            continue

        if sizes.size == 0:
            continue

        t_eff = min(sizes.shape[0], T_MAX)
        frame_sizes[i, :t_eff, cam - 1] = torch.from_numpy(sizes[:t_eff])
        sample_filled = True

        if sizes.shape[0] > T_MAX:
            truncated += 1
        elif sizes.shape[0] < T_MAX:
            shorter += 1

    if sample_filled:
        filled_count += 1

# 4) Report
print(f"Built frame_sizes with E={E}, T_MAX={T_MAX}")
print(f"frame_sizes.shape = {tuple(frame_sizes.shape)}")
print(f"Filled {filled_count} sample(s). Truncated sequences: {truncated}, Shorter-than-T_MAX sequences: {shorter}")
print(f"Test print of the first frame size row: {frame_sizes[0,:10,:]}")

if missing_samples:
    print(f"[INFO] Missing sample_i folders (holes in indices): {sorted(missing_samples)}")

for cam in (1, 2, 3, 4):
    if missing_files[cam]:
        print(f"[WARN] Missing {FILENAME_FMT.format(cam=cam)} for samples: {sorted(missing_files[cam])}")

if bad_files:
    print("[WARN] Unreadable CSVs:")
    for i, cam, err in bad_files:
        print(f"  sample_{i}, camera{cam}: {err}")


Built frame_sizes with E=1010, T_MAX=480
frame_sizes.shape = (1010, 480, 4)
Filled 1010 sample(s). Truncated sequences: 0, Shorter-than-T_MAX sequences: 3948
Test print of the first frame size row: tensor([[183904., 203197., 195331., 201747.],
        [  1982.,   9426.,   4205.,   8901.],
        [  2574.,  10766.,   5423.,  10879.],
        [  5447.,  12535.,   9713.,  12445.],
        [  5508.,  14230.,  10057.,  18284.],
        [  5944.,  14972.,  11136.,  15041.],
        [ 11223.,  15332.,   8558.,  16714.],
        [  7426.,  14382.,  17315.,  15569.],
        [  6984.,  15265.,  13881.,  16640.],
        [ 14634.,  14852.,  12568.,  16781.]])


## Create Packetsize Tensor

In [10]:
# JUPYTER CELL — Build packet-burst tensors from *_pktsize_aggregated.csv
# Produces:
#   pkt_sizes  : [E, T_MAX, 4]  (pkt_frame_sum.len per burst per camera)
#   pkt_times  : [E, T_MAX, 4]  (packet_burst.time_relative per burst per camera)
#   pkt_mask   : [E, T_MAX, 4]  (1 where valid, 0 where padded/missing)
#
# Assumes you've already run the aggregation/padding step so that, within each
# sample_i, all cameras have the same number of bursts (but different samples
# can still have different lengths). We still guard for missing files/rows.

# --- CONFIG (edit these) ---
T_MAX        = 480                                  # desired fixed timeline length
DTYPE        = torch.float32                        # tensor dtype
CAMS         = (1, 2, 3, 4)
FILENAME_FMT = "video_camera{cam}_pktsize_windowed.csv"
# ---------------------------

# 1) Discover sample_i folders (i >= 0) and align E with the largest index
pat = re.compile(r"^sample_(\d+)$")
sample_dirs = {}
for p in PACKETSIZES_DST_ROOT.iterdir():
    if p.is_dir():
        m = pat.match(p.name)
        if m:
            i = int(m.group(1))
            if i >= 0:
                sample_dirs[i] = p

if not sample_dirs:
    raise RuntimeError(f"No sample_i folders found in {PACKETSIZES_DST_ROOT}")

E = max(sample_dirs.keys()) + 1  # tensor index aligns with sample_i (holes remain zeros)

# 2) Allocate tensors
pkt_sizes = torch.zeros((E, T_MAX, len(CAMS)), dtype=DTYPE)             # bytes per burst
pkt_times = torch.zeros((E, T_MAX, len(CAMS)), dtype=DTYPE)             # seconds from pcap start
pkt_mask  = torch.zeros((E, T_MAX, len(CAMS)), dtype=DTYPE)             # 1=valid, 0=missing/padded

# Stats
missing_samples = []
missing_files = {cam: [] for cam in CAMS}
bad_files = []
filled_count = 0
truncated = 0
shorter = 0

def _read_aggregated_csv(csv_path: Path):
    """
    Expect columns: packet_burst.time_relative, pkt_frame_sum.len
    Be tolerant to header/no-header variants.
    Returns two numpy arrays (times, sizes) sorted by time, numeric only.
    """
    # Peek first line for header detection
    try:
        with csv_path.open("r", encoding="utf-8", errors="ignore") as f:
            first = f.readline().strip().lower()
        has_header = ("packet_burst.time_relative" in first) or ("pkt_frame_sum.len" in first)
    except Exception:
        has_header = True  # safe default

    df = pd.read_csv(
        csv_path,
        header=0 if has_header else None,
        names=None if has_header else ["packet_burst.time_relative", "pkt_frame_sum.len"],
        usecols=[0, 1],
        comment="#",
        on_bad_lines="skip",
        dtype=str,
        encoding="utf-8-sig",
        skip_blank_lines=True,
    )

    # Normalize potential alternative header names
    cols = [c.strip().lower() for c in df.columns]
    df.columns = cols
    # Accept a couple of variants just in case
    rename_map = {}
    if "packet_burst.time_relative" not in df.columns and "time" in df.columns:
        rename_map["time"] = "packet_burst.time_relative"
    if "pkt_frame_sum.len" not in df.columns and "size" in df.columns:
        rename_map["size"] = "pkt_frame_sum.len"
    if rename_map:
        df = df.rename(columns=rename_map)

    if not {"packet_burst.time_relative", "pkt_frame_sum.len"} <= set(df.columns):
        raise ValueError(f"Unexpected columns in {csv_path.name}: {list(df.columns)}")

    # Coerce to numeric, drop bad rows, sort by time
    df["packet_burst.time_relative"] = pd.to_numeric(df["packet_burst.time_relative"].astype(str).str.strip(), errors="coerce")
    df["pkt_frame_sum.len"] = pd.to_numeric(df["pkt_frame_sum.len"].astype(str).str.strip(), errors="coerce")
    df = df.dropna(subset=["packet_burst.time_relative", "pkt_frame_sum.len"]).sort_values("packet_burst.time_relative")

    t = df["packet_burst.time_relative"].to_numpy(dtype=float)
    s = df["pkt_frame_sum.len"].to_numpy(dtype=float)
    return t, s

# 3) Fill per sample and per camera
for i in range(E):
    sdir = sample_dirs.get(i)
    if sdir is None:
        missing_samples.append(i)
        continue

    sample_filled = False
    for c_idx, cam in enumerate(CAMS):
        csv_path = sdir / FILENAME_FMT.format(cam=cam)
        if not csv_path.exists():
            missing_files[cam].append(i)
            continue

        try:
            times, sizes = _read_aggregated_csv(csv_path)
        except Exception as e:
            bad_files.append((i, cam, str(e)))
            continue

        if sizes.size == 0:
            continue

        # limit or pad into fixed T_MAX window
        t_eff = min(sizes.shape[0], T_MAX)
        pkt_sizes[i, :t_eff, c_idx] = torch.from_numpy(sizes[:t_eff]).to(DTYPE)
        pkt_times[i, :t_eff, c_idx] = torch.from_numpy(times[:t_eff]).to(DTYPE)
        pkt_mask[i, :t_eff, c_idx] = 1.0
        sample_filled = True

        if sizes.shape[0] > T_MAX:
            truncated += 1
        elif sizes.shape[0] < T_MAX:
            shorter += 1

    if sample_filled:
        filled_count += 1

# 4) Report
print(f"Built pkt_sizes/pkt_times with E={E}, T_MAX={T_MAX}, cams={len(CAMS)}")
print(f"pkt_sizes.shape = {tuple(pkt_sizes.shape)} | dtype={pkt_sizes.dtype}")
print(f"pkt_times.shape = {tuple(pkt_times.shape)} | dtype={pkt_times.dtype}")
print(f"pkt_mask.shape  = {tuple(pkt_mask.shape)}  | dtype={pkt_mask.dtype}")
print(f"Filled {filled_count} sample(s). Truncated sequences: {truncated}, Shorter-than-T_MAX sequences: {shorter}")
print(f"Test print of the first aggregated packet size row: {pkt_times[0,:10,:]}")

if missing_samples:
    print(f"[INFO] Missing sample_i folders (holes in indices): {sorted(missing_samples)}")

for cam in CAMS:
    if missing_files[cam]:
        print(f"[WARN] Missing {FILENAME_FMT.format(cam=cam)} for samples: {sorted(missing_files[cam])}")

if bad_files:
    print("[WARN] Unreadable aggregated CSVs:")
    for i, cam, err in bad_files:
        print(f"  sample_{i}, camera{cam}: {err}")

Built pkt_sizes/pkt_times with E=1010, T_MAX=480, cams=4
pkt_sizes.shape = (1010, 480, 4) | dtype=torch.float32
pkt_times.shape = (1010, 480, 4) | dtype=torch.float32
pkt_mask.shape  = (1010, 480, 4)  | dtype=torch.float32
Filled 1010 sample(s). Truncated sequences: 0, Shorter-than-T_MAX sequences: 4040
Test print of the first aggregated packet size row: tensor([[0.0190, 0.0198, 0.0181, 0.0189],
        [0.0523, 0.0532, 0.0515, 0.0523],
        [0.0857, 0.0865, 0.0848, 0.0856],
        [0.1190, 0.1198, 0.1181, 0.1189],
        [0.1523, 0.1532, 0.1515, 0.1523],
        [0.1857, 0.1865, 0.1848, 0.1856],
        [0.2190, 0.2198, 0.2181, 0.2189],
        [0.2523, 0.2532, 0.2515, 0.2523],
        [0.2857, 0.2865, 0.2848, 0.2856],
        [0.3190, 0.3198, 0.3181, 0.3189]])


## Create Meta Tensor

In [11]:
# Create the tensor for meta values
    # Brand: brand/model of the vehicle {0,1,2}
        # 0. Mercedes Sprinter
        # 1. Nissan Patrol
        # 2. Tesla Model3
    # Color: {0,1,2}
        # 0. Black
        # 1. Gray
        # 2. White
    # Yaw: {-90,0,90,180}
    # vx
    # vy
    # x
    # y
    # rows: Effective length of sample-i

# JUPYTER CELL — Build [E,8] exp_meta_tensor from sample_i/meta.txt files

from pathlib import Path
import re
import pandas as pd
import torch

# --- CONFIG (edit this) ---
DTYPE        = torch.float32
# --------------------------

# Mappings
BRAND_MODEL_MAP = {
    ("mercedes", "sprinter"): 0,
    ("nissan",   "patrol")  : 1,
    ("tesla",    "model3")  : 2,
}
COLOR_MAP = {
    "black": 0,
    "gray":  1, "grey": 1,   # allow both spellings
    "white": 2,
}

# Regex for sample_i and original folder name in meta (to extract brand/model)
SAMPLE_PAT = re.compile(r"^sample_(\d+)$")
# FOLDER_RE  = re.compile(
#     r"""^vehicle\.
#         (?P<brand>[^.]+)\.
#         (?P<model>[^_]+)_
#         (?P<color>[^_]+)_
#         vx(?P<vx>[-+]?\d+(?:\.\d+)?)_
#         vy(?P<vy>[-+]?\d+(?:\.\d+)?)_
#         x(?P<x>[-+]?\d+(?:\.\d+)?)_
#         y(?P<y>[-+]?\d+(?:\.\d+)?)_
#         yaw(?P<yaw>[-+]?\d+(?:\.\d+)?)
#         # _lane(?P<lane>-?\d+)
#         # _randomoffset(?P<offset>[-+]?\d+(?:\.\d+)?)
#         $""",
#     re.VERBOSE
# )

def _read_meta(meta_path: Path) -> dict:
    meta = {}
    if not meta_path.exists():
        return meta
    for line in meta_path.read_text().splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip().lower()] = v.strip()
    return meta

# Discover sample_i folders and align E to max index + 1 (holes become zero rows)
sample_dirs = {}
for p in FRAMESIZES_DST_ROOT.iterdir():
    if p.is_dir():
        m = SAMPLE_PAT.match(p.name)
        if m:
            sample_dirs[int(m.group(1))] = p

if not sample_dirs:
    raise RuntimeError(f"No sample_i folders found in {FRAMESIZES_DST_ROOT}")

E = max(sample_dirs.keys()) + 1
exp_meta_tensor = torch.zeros((E, 8), dtype=DTYPE)

unknown_brand_models = []
unknown_colors = []
missing_meta = []
filled = 0

for i in range(E):
    sdir = sample_dirs.get(i)
    if sdir is None:
        continue
    meta = _read_meta(sdir / "meta.txt")
    if not meta:
        missing_meta.append(i)
        continue

    # Parse brand/model for brand_id (prefer source_folder for model)
    brand_lower = (meta.get("brand") or "").strip().lower()
    color_lower = (meta.get("color") or "").strip().lower()

    brand_id = -1
    # Try to parse brand+model from the source folder line
    src_name = meta.get("source_folder")
    if src_name:
        m = FOLDER_RE.match(src_name)
        if m:
            b = m.group("brand").lower()
            mdl = m.group("model").lower()
            brand_id = BRAND_MODEL_MAP.get((b, mdl), -1)
    # Fallback: if no source_folder or regex failed, try mapping by brand only (unlikely desired)
    if brand_id == -1 and brand_lower:
        # heuristic fallback (can be customized if needed)
        fallback = {
            "mercedes": ("mercedes", "sprinter"),
            "nissan":   ("nissan", "patrol"),
            "tesla":    ("tesla", "model3"),
        }
        key = fallback.get(brand_lower)
        if key:
            brand_id = BRAND_MODEL_MAP.get(key, -1)

    if brand_id == -1:
        unknown_brand_models.append((i, src_name or brand_lower))

    # Color id
    color_id = COLOR_MAP.get(color_lower, -1)
    if color_id == -1:
        unknown_colors.append((i, color_lower))

    # Numeric fields
    def _f(key, default=0.0):
        try:
            return float(meta.get(key, default))
        except Exception:
            return float(default)

    yaw  = _f("yaw", 0.0)   # expected in {-90, 0, 90, 180} but keep as-is
    vx   = _f("vx", 0.0)
    vy   = _f("vy", 0.0)
    x    = _f("x", 0.0)
    y    = _f("y", 0.0)

    # rows: effective length; use 'rows' from meta if present, else 0
    try:
        rows = float(int(float(meta.get("rows", 0))))
    except Exception:
        rows = 0.0

    exp_meta_tensor[i] = torch.tensor([brand_id, color_id, yaw, vx, vy, x, y, rows], dtype=DTYPE)
    filled += 1

print(f"Built exp_meta_tensor with shape {tuple(exp_meta_tensor.shape)} (filled {filled} / E={E})")

if missing_meta:
    print(f"[WARN] Missing meta.txt for samples: {sorted(missing_meta)}")
if unknown_brand_models:
    print("[WARN] Unknown brand/model combos (sample_i, source_folder_or_brand):")
    for tup in unknown_brand_models[:10]:
        print(" ", tup, "...")
    if len(unknown_brand_models) > 10:
        print(f"  ... and {len(unknown_brand_models)-10} more")
if unknown_colors:
    print("[WARN] Unknown colors (sample_i, color):", unknown_colors[:10])

# Optional: quick preview as DataFrame
try:
    import pandas as pd
    cols = ["brand_id","color_id","yaw","vx","vy","x","y","rows"]
    df = pd.DataFrame(exp_meta_tensor.numpy(), columns=cols)
    display(df.head(10))
except Exception:
    pass

# Optional: save to disk
# torch.save({"exp_meta_tensor": exp_meta_tensor}, SAMPLES_DST_ROOT / "exp_meta_tensor.pt")




Built exp_meta_tensor with shape (1010, 8) (filled 1010 / E=1010)


,brand_id,color_id,yaw,vx,vy,x,y,rows
0,0.0,0.0,180.0,-10.525,0.0,-9.475,17.0,269.0
1,0.0,0.0,180.0,-10.670,0.0,-9.330,17.0,269.0
2,0.0,0.0,180.0,-10.700,0.0,-9.300,17.0,269.0
3,0.0,0.0,180.0,-12.083,0.0,-7.917,17.0,239.0
4,0.0,0.0,180.0,-12.972,0.0,-7.028,17.0,239.0
5,0.0,0.0,180.0,-13.042,0.0,-6.958,17.0,239.0
6,0.0,0.0,180.0,-13.066,0.0,-6.934,17.0,239.0
7,0.0,0.0,180.0,-13.112,0.0,-6.888,17.0,239.0
8,0.0,0.0,180.0,-13.385,0.0,-6.615,17.0,239.0
9,0.0,0.0,180.0,-13.471,0.0,-6.529,17.0,239.0


## Save the meta tensors somewhere

In [12]:
# JUPYTER CELL — Save tensors to a given directory (now includes packet sizes)

from pathlib import Path
import torch

# --- CONFIG: set your output directory here ---
# ---------------------------------------------

TENSORS_DST_ROOT.mkdir(parents=True, exist_ok=True)

packet_sizes_dst_root = TENSORS_DST_ROOT/"packetsizes"/network_configuration
packet_sizes_dst_root.mkdir(parents=True, exist_ok=True)

# Collect tensors from the notebook namespace (if they exist)
to_save = {
    "position_gt_tensor.pt":    globals().get("position", None),
    "velocity_gt_tensor.pt":    globals().get("velocity", None),
    "frame_size_tensor.pt":     globals().get("frame_sizes", None),
    "meta_data_tensor.pt":      globals().get("exp_meta_tensor", None),
    # "packet_size_tensor.pt":    globals().get("pkt_sizes", None),   # <— NEW
}

saved = []
missing = []

for fname, tensor in to_save.items():
    if isinstance(tensor, torch.Tensor):
        path = TENSORS_DST_ROOT / fname
        torch.save(tensor.detach().cpu(), path)
        print(f"[OK] Saved {fname}  shape={tuple(tensor.shape)}  dtype={tensor.dtype}")
        saved.append(str(path))
    else:
        print(f"[WARN] Not found or not a torch.Tensor: {fname.split('.pt')[0].replace('_',' ')} variable")
        missing.append(fname)


torch.save(globals().get("pkt_sizes", None).detach().cpu(), packet_sizes_dst_root/"windowed_packet_size_tensor.pt")

print("\nSummary:")
print(" Saved:", len(saved)+1, "files")
if missing:
    print(" Missing:", ", ".join(missing))
print(f" Output directory: {TENSORS_DST_ROOT}")


[OK] Saved position_gt_tensor.pt  shape=(1010, 480, 3)  dtype=torch.float32
[OK] Saved velocity_gt_tensor.pt  shape=(1010, 480, 3)  dtype=torch.float32
[OK] Saved frame_size_tensor.pt  shape=(1010, 480, 4)  dtype=torch.float32
[OK] Saved meta_data_tensor.pt  shape=(1010, 8)  dtype=torch.float32

Summary:
 Saved: 5 files
 Output directory: /home/gaofeng/zanoria/grayassets_datasets/seed1/separatevehicles_3types_3colors_onlyIPframes_720p_no_trafficlights_no_weather_more_redundancy/tensors
